# Phase 1: Targeted Data Cleaning & Preprocessing
This notebook executes the data cleaning pipeline mapped for E-Commerce edge cases.

## Initialization and Loading Data
Load the raw dataset.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("Loading dataset...")
df = pd.read_excel("DailySales_original.xlsx")
print(f"Original shape: {df.shape}")

Loading dataset...
Original shape: (13324, 37)


## 1. Drop the Grand Total Row
Identify and drop the final aggregate row to prevent data duplication.

In [2]:
# Dropping the Grand Total row (indicated by missing Date)
df = df.dropna(subset=["Date"])
print(f"Shape after dropping Grand Total: {df.shape}")

Shape after dropping Grand Total: (13323, 37)


## Basic Cleaning
Dropping columns heavily skewed with nulls that provide no analytical value.

In [3]:
useless_cols = ["Internal Name", "Product Group", "FNSKU", "Is Parent"]
df = df.drop(columns=[col for col in useless_cols if col in df.columns], errors="ignore")
print("Dropped extraneous columns.")

Dropped extraneous columns.


## 2. Smart Imputation for Missing Values
Handling heavy missing values intelligently rather than using crude `.fillna(0)`.

### A. Parent ASIN & Brand/Title Lookup
Fill heavy nulls by mapping them to their base `ASIN` and mapping `Title`/`Brand` from dates where data is present.

In [4]:
# Drop rows where both ASIN and Parent ASIN are null
if "ASIN" in df.columns and "Parent ASIN" in df.columns:
    before_count = df.shape[0]
    df = df.dropna(subset=["ASIN", "Parent ASIN"], how="all")
    print(f"Dropped {before_count - df.shape[0]} rows where both ASIN and Parent ASIN were null.")

# Fill Parent ASIN with ASIN if missing
if "Parent ASIN" in df.columns and "ASIN" in df.columns:
    df["Parent ASIN"] = df["Parent ASIN"].fillna(df["ASIN"])

# Map missing strings via SKU, ASIN, and Parent ASIN references
for col in ["Title", "Brand"]:
    if col in df.columns:
        # 1. SKU Map
        if "SKU" in df.columns:
            sku_map = df.dropna(subset=[col]).groupby("SKU")[col].first()
            df[col] = df[col].fillna(df["SKU"].map(sku_map))
        # 2. ASIN Map
        if "ASIN" in df.columns:
            asin_map = df.dropna(subset=[col]).groupby("ASIN")[col].first()
            df[col] = df[col].fillna(df["ASIN"].map(asin_map))
        # 3. Parent ASIN Map
        if "Parent ASIN" in df.columns:
            pasin_map = df.dropna(subset=[col]).groupby("Parent ASIN")[col].first()
            df[col] = df[col].fillna(df["Parent ASIN"].map(pasin_map))
        # 4. Failsafe for remaining nulls
        df[col] = df[col].fillna(f"Unknown {col}")
print("Parent ASIN and Categorical maps imputed and nulls fixed.")


Parent ASIN and Categorical map imputed.


### B. Per Unit Revenue Imputation
Calculate historical average price for specific SKUs, or set to 0 if 0 orders are placed.

In [5]:
if "Per Unit Revenue" in df.columns and "Orders" in df.columns:
    # Calculate historical average price for specific SKU
    avg_price_map = df[df["Per Unit Revenue"] > 0].groupby("SKU")["Per Unit Revenue"].mean()
    
    # Fill with historical average first
    df["Per Unit Revenue"] = df["Per Unit Revenue"].fillna(df["SKU"].map(avg_price_map))
    
    # Failsafe to 0 for missing limits and 0 orders
    df["Per Unit Revenue"] = np.where(
        df["Per Unit Revenue"].isna() & (df["Orders"] == 0),
        0,
        df["Per Unit Revenue"]
    )
    df["Per Unit Revenue"] = df["Per Unit Revenue"].fillna(0)
print("Per Unit Revenue securely imputed.")

Per Unit Revenue securely imputed.


## 3. Formatting & Export
Formatting dates and deriving structural Gross Margins before saving.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(0)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

if "Net Profit" in df.columns and "Revenue" in df.columns:
    df["Gross Margin"] = df["Revenue"] - df["COGS"].abs()

output_file = "DailySales_cleaned_professional.xlsx"
df.to_excel(output_file, index=False)
print(f"Fully cleaned dataset saved to {output_file}")